In [30]:
import torch
import gc

# 1. 删除可能占用大量显存的变量
#    确保这些变量名与您代码中实际使用的变量名一致
#    如果 'model' 或 'trainer' 等对象不存在，尝试删除它们会报错，
#    所以我们先检查它们是否存在。

if 'model' in locals() or 'model' in globals():
    try:
        del model
        print("变量 'model' 已删除。")
    except NameError:
        pass # 变量未定义

if 'trainer' in locals() or 'trainer' in globals():
    try:
        del trainer
        print("变量 'trainer' 已删除。")
    except NameError:
        pass # 变量未定义

# 如果还有其他大型数据结构或模型组件，也一并删除
# 例如，如果您有旧的 optimizer 或 dataset 对象：
# if 'optimizer' in locals(): del optimizer
# if 'formatted_train_dataset' in locals(): del formatted_train_dataset # 如果这个数据集很大且不再直接需要

# 2. 执行 Python 的垃圾回收
#    这会尝试回收不再被引用的对象的内存。
gc.collect()
print("Python 垃圾回收已执行。")

# 3. 清理 PyTorch 的 CUDA 缓存
#    这会释放 PyTorch 在 CUDA 上缓存但未被占用的显存。
#    只有在 PyTorch 和 CUDA 可用时才执行。
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("PyTorch CUDA 缓存已清理。")
else:
    print("CUDA 不可用，跳过 CUDA 缓存清理。")

print("\n显存清理尝试完成。现在您可以应用新的配置并重新开始训练。")

Python 垃圾回收已执行。
PyTorch CUDA 缓存已清理。

显存清理尝试完成。现在您可以应用新的配置并重新开始训练。


In [17]:
from datasets import load_dataset
import re
import os

local_dataset_dir = "./data/tang_poems/" 

dataset_files = {
    'test': os.path.join(local_dataset_dir, 'test-00000-of-00001-a794cd4c018c9326.parquet'),
    'train': os.path.join(local_dataset_dir, 'train-00000-of-00001-6914ee5fabc145c0.parquet')
}

all_splits = load_dataset('parquet', data_files=dataset_files)

train_dataset = all_splits['train']
test_dataset = all_splits['test']


def format_poetry_prompt(sample):
    """
    处理单条数据，将其格式化为模型输入格式。
    无效数据将返回 {"messages": None}。
    """
    poetry_column_name = 'paragraphs'
    
    # 修改点1：无效时返回 {"messages": None}
    if poetry_column_name not in sample or sample[poetry_column_name] is None:
        return {"messages": None} 
        
    raw_poem_data = sample[poetry_column_name]
    
    if isinstance(raw_poem_data, list):
        poem_text = "".join(raw_poem_data) 
    elif isinstance(raw_poem_data, str):
        poem_text = raw_poem_data
    else:
        return {"messages": None} # 修改点2

    poem_text = poem_text.strip()
    if not poem_text:
        return {"messages": None} # 修改点3

    parts = re.split(r'([，。？！])', poem_text)
    if len(parts) < 3:
        return {"messages": None} # 修改点4

    prompt_starter = parts[0] + parts[1]
    completion = "".join(parts[2:]).strip()
    
    if not prompt_starter or not completion:
        return {"messages": None} # 修改点5

    # 成功处理的情况
    messages_content = [
        {"role": "user", "content": f"请补全这首唐诗：{prompt_starter}"},
        {"role": "assistant", "content": completion}
    ]
    return {"messages": messages_content} # 保持不变

# 使用map函数处理训练数据集
formatted_train_dataset = train_dataset.map(
    format_poetry_prompt, 
    remove_columns=train_dataset.column_names 
)


formatted_train_dataset = formatted_train_dataset.filter(
    lambda x: x.get("messages") is not None
)


if len(formatted_train_dataset) > 0:
    print("数据格式示例 (训练集):")
    print(formatted_train_dataset[0]['messages'])
    print(f"\n处理后的训练集样本数量: {len(formatted_train_dataset)}")
else:
    print("警告：处理后的训练数据集为空，请检查 format_poetry_prompt 函数、原始数据或 'paragraphs' 列的内容。")
    

数据格式示例 (训练集):
[{'content': '请补全这首唐诗：东宫白庶子，', 'role': 'user'}, {'content': '南寺远禅师。何处遥相见，心无一事时。', 'role': 'assistant'}]

处理后的训练集样本数量: 35998


In [18]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- 修改开始 ---
# 将模型ID修改为本地模型文件所在的目录路径
local_model_path = "./pretrain/Qwen1.5-1.8B-Chat/"
# --- 修改结束 ---

# QLoRA 量化配置 (这部分保持不变)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # 在支持的硬件上使用bfloat16以获得更好性能
)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    local_model_path, # <--- 修改这里，使用本地路径
    quantization_config=bnb_config,
    device_map="auto" # 自动将模型分发到可用设备（GPU）
)

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(local_model_path) # <--- 修改这里，使用本地路径
# Qwen1.5没有默认的pad_token，我们通常可以将其设置为eos_token
tokenizer.pad_token = tokenizer.eos_token

print(f"模型和分词器已从本地路径 '{local_model_path}' 加载。")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


模型和分词器已从本地路径 './pretrain/Qwen1.5-1.8B-Chat/' 加载。


In [19]:
from peft import LoraConfig, get_peft_model


lora_config = LoraConfig(
    r=16,  # LoRA的秩，一个关键超参数
    lora_alpha=32, # LoRA的alpha，通常是r的两倍
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ], # 指定要应用LoRA的模块
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 将LoRA适配器应用到模型上
model = get_peft_model(model, lora_config)

# 打印可训练参数，验证LoRA是否生效
model.print_trainable_parameters()
# 输出会显示可训练参数远小于总参数，例如:
# trainable params: 10,502,656 || all params: 1,847,199,744 || trainable%: 0.5685


from transformers import TrainingArguments
from trl import SFTTrainer

# (Your existing code for model, tokenizer, bnb_config, lora_config, formatted_train_dataset)
# tokenizer = AutoTokenizer.from_pretrained(...) # Ensure this is defined
# tokenizer.pad_token = tokenizer.eos_token # Ensure this is set

# Define formatting_func as shown above
def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return text

training_args = TrainingArguments(
    output_dir="./results_qwen1.5_poetry",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    optim="paged_adamw_8bit",
    logging_steps=20,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    warmup_ratio=0.03,
    bf16=True, # Or fp16=True if bf16 is not supported
    report_to="wandb",
    run_name="qwen1.5-1.8b-poetry-sft-run-1"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_train_dataset, # Your dataset with the 'messages' field
    peft_config=lora_config,
    dataset_text_field="messages",         # You can keep this, but formatting_func takes precedence
                                           # for ConstantLengthDataset's text processing logic.
                                           # Alternatively, set to None if formatting_func is robust.
                                           # For clarity, keeping it shows where the 'messages' are.
    formatting_func=formatting_func,       # <--- Add your formatting function here
    max_seq_length=1024,
    args=training_args,
    packing=True                           # Keep packing=True
)

print("开始微调...")
trainer.train()
print("微调完成！")

trainer.save_model("./pretrain/final_poetry_adapter")
print("模型适配器已保存到 ./pretrain/final_poetry_adapter")


trainable params: 14,991,360 || all params: 1,851,820,032 || trainable%: 0.8095473502254457
开始微调...


Step,Training Loss
20,3.084900
40,2.577500
60,2.547100
80,2.534600
100,2.512800
120,2.506400
140,2.520900


/root/miniconda3/lib/python3.12/site-packages/peft/utils/save_and_load.py:154: UserWarning: Could not find a config file in ./pretrain/Qwen1.5-1.8B-Chat/ - will assume that the vocabulary was not modified.
  warnings.warn(


微调完成！


/root/miniconda3/lib/python3.12/site-packages/peft/utils/save_and_load.py:154: UserWarning: Could not find a config file in ./pretrain/Qwen1.5-1.8B-Chat/ - will assume that the vocabulary was not modified.
  warnings.warn(


模型适配器已保存到 ./pretrain/final_poetry_adapter


In [26]:

from transformers import pipeline
import torch

# 加载我们刚刚训练好的模型适配器进行推理
# 在SFTTrainer训练后，模型已在内存中准备好，可以直接使用
# 如果要从头加载，请参考下面的注释代码

# --- 如果在新的脚本中加载模型 ---
from peft import AutoPeftModelForCausalLM
trained_model = AutoPeftModelForCausalLM.from_pretrained(
    "./pretrain/final_poetry_adapter",
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
trained_tokenizer = AutoTokenizer.from_pretrained("./pretrain/final_poetry_adapter")
# ------------------------------------

# # 直接使用trainer中的模型和分词器
# trained_model = trainer.model
# trained_tokenizer = trainer.tokenizer

# 定义几个测试开头
prompts = [
    "白日依山尽，",
    "红豆生南国，",
    "床前明月光，", # 一个非常经典的例子
    "双燕东南飞，",
    "万里扬帆迎六月，"
]

for start_text in prompts:
    # 构造与训练时一致的输入格式
    messages = [
        {"role": "user", "content": f"请补全这首唐诗：{start_text}"}
    ]
    
    # 使用pipeline进行生成
    pipe = pipeline("text-generation", model=trained_model, tokenizer=trained_tokenizer)
    
    # apply_chat_template 会将消息列表转换为模型能理解的单个字符串
    prompt_for_model = trained_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    print(f"--- 测试开头: {start_text} ---")
    
    outputs = pipe(
        prompt_for_model,
        max_new_tokens=60, # 生成的最大token数
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        eos_token_id=trained_tokenizer.eos_token_id,
        pad_token_id=trained_tokenizer.pad_token_id
    )
    
    generated_text = outputs[0]['generated_text']
    # 从生成结果中提取助手的回复
    assistant_response = generated_text.split("<|im_start|>assistant")[1].replace("<|im_end|>", "").strip()
    
    print(f"模型生成结果:\n{start_text}{assistant_response}\n")



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCaus

--- 测试开头: 白日依山尽， ---


The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalLM', 'MusicgenMelodyFo

模型生成结果:
白日依山尽，黄河入海流。欲穷千里目，更上一层楼。

--- 测试开头: 红豆生南国， ---


The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalLM', 'MusicgenMelodyFo

模型生成结果:
红豆生南国，春来发几枝。愿君多采撷，此物最相思。

--- 测试开头: 床前明月光， ---


The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalLM', 'MusicgenMelodyFo

模型生成结果:
床前明月光，疑是地上霜。举头望明月，低头思故乡。

--- 测试开头: 双燕东南飞， ---


The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalLM', 'MusicgenMelodyFo

模型生成结果:
双燕东南飞，一蝉北去吟。欲知相问处，应是陇头林。

--- 测试开头: 万里扬帆迎六月， ---
模型生成结果:
万里扬帆迎六月，千山尽入沧波里。风高夜半江声绝，空有碧天如水色。一叶扁舟横一叶，愁看明月照孤舟。



In [31]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, GenerationConfig
from peft import PeftModel, AutoPeftModelForCausalLM
from datasets import load_dataset
import re
import os

# --- 配置路径 ---
base_model_path = "./pretrain/Qwen1.5-1.8B-Chat/"
adapter_path = "./pretrain/final_poetry_adapter/" # 您微调后的适配器路径
local_dataset_dir = "./data/tang_poems/"
test_data_file = os.path.join(local_dataset_dir, 'test-00000-of-00001-a794cd4c018c9326.parquet')
num_test_samples_to_show = 10 # 您想测试并展示的样本数量

# --- 1. 加载分词器 (通常基线模型和微调后的模型使用相同的分词器) ---
# 确保 adapter_path 包含分词器文件，或者从 base_model_path 加载
# 通常，微调适配器时，分词器是基于基座模型的，可以直接从基座模型路径加载，或者PEFT保存时也会保存分词器配置
try:
    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    print(f"Tokenizer loaded from adapter path: {adapter_path}")
except OSError:
    print(f"Could not load tokenizer from adapter path, trying base model path: {base_model_path}")
    tokenizer = AutoTokenizer.from_pretrained(base_model_path)
    print(f"Tokenizer loaded from base model path: {base_model_path}")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Set tokenizer pad_token to eos_token.")

# --- 2. 加载基线模型 (Baseline Model) ---
print("\nLoading BASELINE model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, # 或者 torch.float32 如果bfloat16导致问题
)
baseline_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16, # 匹配 compute_dtype
    low_cpu_mem_usage=True,
)
baseline_model.eval() # 设置为评估模式
print("BASELINE model loaded.")

# --- 3. 加载微调后的模型 (Fine-tuned Model) ---
print("\nLoading FINE-TUNED model...")
# 加载时，它会自动加载基座模型（在adapter_path中的config指定的），然后应用adapter
# 注意：AutoPeftModelForCausalLM.from_pretrained 会先加载基座模型，然后应用LoRA权重。
# 它使用的基座模型路径是在保存适配器时，通过 config.json 中的 base_model_name_or_path 指定的。
# 请确保该路径指向您实际的基座模型（例如 ./pretrain/Qwen1.5-1.8B-Chat/）
# 或者，我们可以先加载基座模型，再加载PEFT适配器，这样更明确：

# 先加载基座模型实例（可以复用上面的 bnb_config）
# 注意：如果使用 AutoPeftModelForCausalLM，它内部会加载基座。
# 为确保加载的是正确的量化配置，我们手动加载基座再应用PEFT。
base_for_finetuned = AutoModelForCausalLM.from_pretrained(
    base_model_path, # 确保这个路径是正确的基座模型路径
    quantization_config=bnb_config, # 应用同样的量化配置
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
# 然后加载LoRA权重到这个基座模型上
fine_tuned_model = PeftModel.from_pretrained(base_for_finetuned, adapter_path)
# 可选：合并权重以加速推理（但会消耗更多内存，且模型变为非PeftModel）
# fine_tuned_model = fine_tuned_model.merge_and_unload()
fine_tuned_model.eval() # 设置为评估模式
print("FINE-TUNED model loaded.")


# --- 4. 加载并准备测试数据集 ---
print(f"\nLoading test dataset from: {test_data_file}")
try:
    test_dataset_full = load_dataset("parquet", data_files={"test": test_data_file}, split="test")
    # 随机选择N个样本进行测试
    test_samples = test_dataset_full.shuffle(seed=42).select(range(num_test_samples_to_show))
    print(f"Loaded {len(test_dataset_full)} samples from test set. Selected {num_test_samples_to_show} for demo.")
except Exception as e:
    print(f"Error loading test dataset: {e}")
    print("Using predefined prompts as fallback.")
    test_samples = [] # 清空，以便使用下面的 prompts

# 如果数据集加载失败或为空，使用预定义的prompts
if not test_samples:
    predefined_prompts = [
        "白日依山尽，",
        "红豆生南国，",
        "床前明月光，",
        "双燕东南飞，",
        "万里扬帆迎六月，"
    ]
    prompts_from_data = [{"starter_text": p, "actual_completion": "[N/A - Predefined Prompt]", "full_original_poem": p + " [N/A - Predefined Prompt]" } for p in predefined_prompts]
    print(f"Using {len(predefined_prompts)} predefined prompts.")
else:
    def get_prompt_from_sample(sample):
        poem_text_list_or_str = sample['paragraphs'] # 假设诗歌在 'paragraphs' 列
        if isinstance(poem_text_list_or_str, list):
            poem_text = "".join(poem_text_list_or_str)
        elif isinstance(poem_text_list_or_str, str):
            poem_text = poem_text_list_or_str
        else:
            return None, None, None # 无效数据

        poem_text = poem_text.strip()
        if not poem_text:
            return None, None, None

        parts = re.split(r'([，。？！])', poem_text)
        if len(parts) < 3: # 诗歌太短，无法分割
            return None, None, None

        starter = parts[0] + parts[1]
        completion = "".join(parts[2:]).strip()
        return starter, completion, poem_text

    prompts_from_data = []
    for sample in test_samples:
        starter, actual_completion, full_poem = get_prompt_from_sample(sample)
        if starter:
            prompts_from_data.append({
                "starter_text": starter,
                "actual_completion": actual_completion,
                "full_original_poem": full_poem
            })
    print(f"Prepared {len(prompts_from_data)} prompts from test dataset.")

# --- 5. 定义生成函数 ---
def generate_response(model_pipeline, prompt_starter_text, tokenizer_ref):
    messages = [{"role": "user", "content": f"请补全这首唐诗：{prompt_starter_text}"}]
    prompt_for_model = tokenizer_ref.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 定义生成参数
    generation_args = {
        "max_new_tokens": 60,
        "do_sample": True,
        "temperature": 0.7,
        "top_k": 50,
        "top_p": 0.95,
        "eos_token_id": tokenizer_ref.eos_token_id,
        "pad_token_id": tokenizer_ref.pad_token_id  # 确保 pad_token_id 被设置
    }

    try:
        outputs = model_pipeline(prompt_for_model, **generation_args)
        generated_text_full = outputs[0]['generated_text']

        # 从生成结果中提取助手的回复
        # Qwen 格式通常是 <|im_start|>system...<|im_end|>\n<|im_start|>user...<|im_end|>\n<|im_start|>assistant\n...<|im_end|>
        # 我们需要 assistant 之后的部分
        assistant_response_match = re.search(r"<\|im_start\|>assistant\n(.*?)(<\|im_end\|>|$)", generated_text_full, re.DOTALL)
        if assistant_response_match:
            assistant_response = assistant_response_match.group(1).strip()
        else: # 如果上面的正则匹配失败，尝试一个更通用的分割方法
            parts = generated_text_full.split("<|im_start|>assistant\n")
            if len(parts) > 1:
                assistant_response = parts[-1].split("<|im_end|>")[0].strip()
            else: # 如果还是不行，可能模板不同或生成不完整
                assistant_response = "[未能解析AI回复]"
                print(f"Warning: Could not parse assistant response from: {generated_text_full}")
        return assistant_response
    except RuntimeError as e:
        if "probability tensor contains either `inf`, `nan` or element < 0" in str(e):
            print(f"RUNTIME ERROR for prompt '{prompt_starter_text}': {e}. Skipping this generation.")
            return "[生成时遇到RuntimeError]"
        else:
            raise e # 重新抛出其他 RuntimeError
    except Exception as e:
        print(f"ERROR during generation for prompt '{prompt_starter_text}': {e}")
        return "[生成时发生错误]"


# --- 6. 创建 Pipelines ---
print("\nCreating pipelines...")
# device_map="auto" 会自动分配到GPU（如果可用）或CPU
# 对于pipeline，也可以直接指定 device=0 (GPU 0) 或 device=-1 (CPU)
# 如果在GPU上运行，确保模型加载时 device_map="auto" 或已 .to(device)
device = 0 if torch.cuda.is_available() else -1 # 自动选择GPU或CPU

pipe_baseline = pipeline("text-generation", model=baseline_model, tokenizer=tokenizer,)
pipe_finetuned = pipeline("text-generation", model=fine_tuned_model, tokenizer=tokenizer,)
print("Pipelines created.")

# --- 7. 循环测试并打印结果 ---
print("\n--- 开始生成对比结果 ---")
for item_data in prompts_from_data:
    start_text = item_data["starter_text"]
    actual_completion_text = item_data["actual_completion"]
    full_original_poem_text = item_data["full_original_poem"]

    print(f"\n======================================================================")
    print(f"📜 测试开头: {start_text}")
    print(f"📖 参考原文: {full_original_poem_text}")
    print(f"----------------------------------------------------------------------")

    print("⏳ 正在使用基线模型生成...")
    baseline_completion = generate_response(pipe_baseline, start_text, tokenizer)
    print(f"🤖 基线模型生成:\n{start_text}{baseline_completion}")
    print(f"----------------------------------------------------------------------")

    print("⏳ 正在使用微调模型生成...")
    finetuned_completion = generate_response(pipe_finetuned, start_text, tokenizer)
    print(f"🚀 微调模型生成:\n{start_text}{finetuned_completion}")
    print(f"======================================================================\n")

print("所有选定样本测试完成。")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer loaded from adapter path: ./pretrain/final_poetry_adapter/

Loading BASELINE model...
BASELINE model loaded.

Loading FINE-TUNED model...
FINE-TUNED model loaded.

Loading test dataset from: ./data/tang_poems/test-00000-of-00001-a794cd4c018c9326.parquet


The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalLM', 'MusicgenMelodyFo

Loaded 5274 samples from test set. Selected 10 for demo.
Prepared 10 prompts from test dataset.

Creating pipelines...
Pipelines created.

--- 开始生成对比结果 ---

📜 测试开头: 廷评年少法家流，
📖 参考原文: 廷评年少法家流，心似澄江月正秋。学究天人知远识，权分盐铁许良筹。春风忆酒乌家近，好月论禅谢寺幽。清白比来谁见赏，怜君独有富人侯。
----------------------------------------------------------------------
⏳ 正在使用基线模型生成...
🤖 基线模型生成:
廷评年少法家流，廷评年少法家流，
深思明理度朝晖。
倡言法治彰正义，
民风淳朴振国威。

论断公平公正，
德政垂范天下人。
律己慎行守公道，
清正廉洁树楷模。

执法如山
----------------------------------------------------------------------
⏳ 正在使用微调模型生成...
🚀 微调模型生成:
廷评年少法家流，玉帐横陈金鼎游。玉匣中分千载雪，琼筵上会九秋风。春来见舞朝倾国，秋至闻歌夜不空。为报主恩还下泪，故园无处问芳踪。


📜 测试开头: 枯木藏龙，
📖 参考原文: 枯木藏龙，雷动必惊。惊者是少，不惊者多。
----------------------------------------------------------------------
⏳ 正在使用基线模型生成...
🤖 基线模型生成:
枯木藏龙，枯木藏龙，独树一帜；  
百年风雨，傲视群雄。  
枝叶苍老，岁月如歌；  
坚韧不拔，风骨犹存。  
繁花似锦，生机勃发；  
枝干参天，山河壮丽。
----------------------------------------------------------------------
⏳ 正在使用微调模型生成...
🚀 微调模型生成:
枯木藏龙，孤松隐士。故人归去，犹有高名。古柏寒声，萧疏阴气。秋风不改，长夜难明。


📜 测试开头: 暗窦养泉容决决，
📖 参考原文: 暗窦养泉容决决，明园护桂放亭亭。历山居处